# Exploration

## Data Collection

In [2]:
import pandas as pd
import numpy as np
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import json

In [ ]:
BASE_URL = "https://api.imdbapi.dev/"
response = requests.get(f"{BASE_URL}/titles?types=MOVIE&startYear>=2024")
data = response.json()
df = pd.DataFrame(data["titles"])
pagetoken = data["nextPageToken"]
time.sleep(1)  

for page in range(1, 20):
      try:
            response = requests.get(f"{BASE_URL}/titles?types=MOVIE&startYear=2026&pageToken={pagetoken}", timeout=10)
            
            if response.status_code == 429:
                  wait_time = int(response.headers.get("Retry-After", 5))
                  print(f"Page {page}: Rate limited! Waiting {wait_time}s...")
                  time.sleep(wait_time)
                  continue
            
            response.raise_for_status()
            
            if not response.text:
                  break
            
            data = response.json()
            
            if "titles" not in data or not data["titles"]:
                  break
                  
            df_page = pd.DataFrame(data["titles"])
            df = pd.concat([df, df_page], ignore_index=True)
            
            if "nextPageToken" in data and data["nextPageToken"]:
                  pagetoken = data["nextPageToken"]
            else:
                  break
            
            time.sleep(1)  
      except Exception as e:
            break

print(f"Final shape: {df.shape}")
print(f"Total Number of rows: {len(df)}")
display(df.head())

Final shape: (1000, 10)
Total Number of rows: 1000


,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot
0,tt12042730,movie,Project Hail Mary,Project Hail Mary,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,9360.0,"[Adventure, Comedy, Sci-Fi]","{'aggregateRating': 8.4, 'voteCount': 178625}",A science teacher wakes up alone on a spaceshi...
1,tt28650488,movie,The Super Mario Galaxy Movie,The Super Mario Galaxy Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5880.0,"[Animation, Adventure, Comedy, Family, Fantasy]","{'aggregateRating': 6.5, 'voteCount': 25522}","Mario ventures into space, exploring cosmic wo..."
2,tt32430579,movie,Crime 101,Crime 101,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,8400.0,"[Crime, Drama, Thriller]","{'aggregateRating': 6.8, 'voteCount': 48321}","An elusive thief, eyeing his final score, enco..."
3,tt33071426,movie,The Drama,The Drama,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6300.0,"[Comedy, Drama, Romance]","{'aggregateRating': 7.5, 'voteCount': 17072}",A happily engaged couple is put to the test wh...
4,tt37209937,movie,Pizza Movie,Pizza Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5820.0,[Comedy],"{'aggregateRating': 5.8, 'voteCount': 4258}",High college students face an unexpectedly epi...


In [ ]:
def fetch_box_office(ind, row, base_url, delay=0.5):
    title_id = row["id"]
    time.sleep(delay)  
    try:
        response = requests.get(f"{base_url}/titles/{title_id}/boxOffice", timeout=5)
        
        if response.status_code == 429:
            wait_time = int(response.headers.get("Retry-After", 10))
            print(f"Rate limited on {title_id}, waiting {wait_time}s...")
            time.sleep(wait_time)
            return fetch_box_office(ind, row, base_url, delay=2)  #
        
        response.raise_for_status() 
        data = response.json()
        worldwide_gross = data["worldwideGross"]["amount"] if "worldwideGross" in data else None
        production_budget = data["productionBudget"]["amount"] if "productionBudget" in data else None
        return ind, worldwide_gross, production_budget, None
    except requests.Timeout:
        return ind, None, None, f"{title_id}: Timeout"
    except requests.HTTPError as e:
        return ind, None, None, f"{title_id}: HTTP {response.status_code}"
    except json.JSONDecodeError:
        return ind, None, None, f"{title_id}: Invalid JSON response"
    except Exception as e:
        return ind, None, None, f"{title_id}: {type(e).__name__}"

skipped_count = 0
for ind, row in df.iterrows():
    ind_result, worldwide_gross, production_budget, error = fetch_box_office(ind, row, BASE_URL)
    if error:
        skipped_count += 1
    else:
        df.at[ind, "worldwideGross"] = worldwide_gross
        df.at[ind, "productionBudget"] = production_budget
    
    if (ind + 1) % 100 == 0:
        print(f"Processed {ind + 1}/{len(df)} titles...")

print(f"Skipped {skipped_count} titles due to errors.")
print(f"Box office data added to {len(df) - skipped_count} rows.")

display(df.head(10))

Processed 100/1000 titles...
Processed 200/1000 titles...
Processed 300/1000 titles...
Processed 400/1000 titles...
Processed 500/1000 titles...
Processed 600/1000 titles...
Processed 700/1000 titles...
Processed 800/1000 titles...
Processed 900/1000 titles...
Processed 1000/1000 titles...
Skipped 0 titles due to errors.
Box office data added to 1000 rows.


In [5]:
display(df.head(10))

,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot,worldwideGross,productionBudget
0,tt12042730,movie,Project Hail Mary,Project Hail Mary,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,9360.0,"[Adventure, Comedy, Sci-Fi]","{'aggregateRating': 8.4, 'voteCount': 178625}",A science teacher wakes up alone on a spaceshi...,433030505,200000000
1,tt28650488,movie,The Super Mario Galaxy Movie,The Super Mario Galaxy Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5880.0,"[Animation, Adventure, Comedy, Family, Fantasy]","{'aggregateRating': 6.5, 'voteCount': 25522}","Mario ventures into space, exploring cosmic wo...",437751829,110000000
2,tt32430579,movie,Crime 101,Crime 101,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,8400.0,"[Crime, Drama, Thriller]","{'aggregateRating': 6.8, 'voteCount': 48321}","An elusive thief, eyeing his final score, enco...",72559167,90000000
3,tt33071426,movie,The Drama,The Drama,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6300.0,"[Comedy, Drama, Romance]","{'aggregateRating': 7.5, 'voteCount': 17072}",A happily engaged couple is put to the test wh...,26232083,None
4,tt37209937,movie,Pizza Movie,Pizza Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5820.0,[Comedy],"{'aggregateRating': 5.8, 'voteCount': 4258}",High college students face an unexpectedly epi...,None,None
5,tt27543632,movie,The Housemaid,The Housemaid,{'url': 'https://m.media-amazon.com/images/M/M...,2025.0,7860.0,"[Drama, Mystery, Thriller]","{'aggregateRating': 6.8, 'voteCount': 128853}",A struggling young woman is relieved by the ch...,399318859,35000000
6,tt33244668,movie,Anaconda,Anaconda,{'url': 'https://m.media-amazon.com/images/M/M...,2025.0,5940.0,"[Action, Adventure, Comedy]","{'aggregateRating': 5.6, 'voteCount': 52429}",A group of friends are going through a mid-lif...,134974943,45000000
7,tt27552099,movie,Mike & Nick & Nick & Alice,Mike & Nick & Nick & Alice,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6420.0,"[Action, Comedy, Crime]","{'aggregateRating': 6.2, 'voteCount': 12231}",Two friends navigate the dangerous world of or...,None,None
8,tt39139925,movie,Dhurandhar The Revenge,Dhurandhar The Revenge,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,13800.0,"[Action, Crime, Thriller]","{'aggregateRating': 8.5, 'voteCount': 43693}",Jaskirat Singh Rangi descends deeper into his ...,150917485,None
9,tt0049833,movie,The Ten Commandments,The Ten Commandments,{'url': 'https://m.media-amazon.com/images/M/M...,1956.0,13200.0,"[Adventure, Drama, Family, History]","{'aggregateRating': 7.9, 'voteCount': 84370}","Moses, raised as a prince of Egypt in the Phar...",65501273,13282712


In [ ]:
missing_counts = df.isna().sum()
print(missing_counts)
missing_counts[missing_counts > 0]